## Imports

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [10]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [11]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [12]:
## Parameter 
start_date = '20240401'
end_date = '20240630'

In [14]:
## base 

base_customer_query = f"""

    SELECT  
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        COUNT(DISTINCT user_id) active_customers
    FROM 
        canonical.clevertap_customer_order_activity 
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        -- AND serviceable = 'true'
        -- AND order_activity_source IN ('registration','login','appOpen') --inOrder
"""

df_base_customer = pd.read_sql(base_customer_query, connection)
df_base_customer

,date_range,active_customers
0,20240401 - 20240630,36599809


36.6M Active Platform Customers Base

### Professional Urban Commuters

In [101]:
def generate_monthly_queries(start_date, end_date):
    queries = []
    current_date = datetime.strptime(start_date, '%Y%m%d')
    end_date = datetime.strptime(end_date, '%Y%m%d')
    
    while current_date <= end_date:
        month_start = current_date.replace(day=1)
        next_month_start = (month_start + timedelta(days=32)).replace(day=1)
        month_end = next_month_start - timedelta(days=1)
        
        month_start_str = month_start.strftime('%Y%m%d')
        month_end_str = month_end.strftime('%Y%m%d')
        
        query = f"""
            SELECT
                customer_id,
                usecase
            FROM 
                (    
                    (
                    SELECT  
                        customer_id,
                        pickup_usecase.usecase
                    FROM
                        orders.order_logs_snapshot ols 
                    JOIN 
                        experiments_internal.combined_usecase_hex_10_level pickup_usecase
                        ON pickup_location_hex_10 = pickup_usecase.hex_10
                        AND usecase IN ('office')
                        
                    WHERE 
                        yyyymmdd >= '{month_start_str}'
                        AND yyyymmdd <= '{month_end_str}'
                    )
                UNION
                    (
                    SELECT  
                        customer_id,
                        drop_usecase.usecase 
                    FROM
                        orders.order_logs_snapshot ols 
                    JOIN 
                        experiments_internal.combined_usecase_hex_10_level drop_usecase
                        ON drop_location_hex_10 = drop_usecase.hex_10
                        AND usecase IN ('office')
                        
                    WHERE 
                        yyyymmdd >= '{month_start_str}'
                        AND yyyymmdd <= '{month_end_str}'
                    )
                UNION
                    (
                    SELECT 
                        identity customer_id,
                        'office' usecase
                    FROM 
                        canonical.clevertap_customer_fare_estimate
                    WHERE 
                        yyyymmdd >= '{month_start_str}'
                        AND yyyymmdd <= '{month_end_str}'
                        AND (pickup_address IS NOT NULL OR drop_address IS NOT NULL)
                        AND 
                            (
                            regexp_like(lower(pickup_address), 'office|corp|ltd|llc|inc|headquarter|tech|solutions|innovations|systems|software|enterprises|business|labs|digital|network')
                            OR
                            regexp_like(lower(drop_address), 'office|corp|ltd|llc|inc|headquarter|tech|solutions|innovations|systems|software|enterprises|business|labs|digital|network')
                            )
                    GROUP BY 1
                    )
                UNION
                    (
                    SELECT 
                        id customer_id,
                        'office' usecase
                    FROM 
                        raw_internal.kafka_info_customer_installed_apps_v1
                    WHERE 
                        yyyymmdd >= '{month_start_str}'
                        AND yyyymmdd <= '{month_end_str}'
                        AND regexp_like(lower(data), 'asana|cisco|confluence|dropbox|figma|github|analytics|authenticator|company|jira|medium|365|office|teams|miro|authenticator|nishtha|onedrive|outlook|slack|slack|trello|webex|wunderlist|zoho|zoho|zoom')                            
                    GROUP BY 1
                    )
                )
            GROUP BY 1,2
        """
        queries.append(query)
        current_date = next_month_start
    
    return queries

In [102]:
monthly_queries = generate_monthly_queries(start_date, end_date)

# Execute each query and collect the results
df_list = [pd.read_sql(query, connection) for query in monthly_queries]

# Concatenate all dataframes and drop duplicates
df_customer_usecase = pd.concat(df_list).drop_duplicates()

# Group by usecase and count unique customer_ids
df_customer_use_case = df_customer_usecase.groupby(['usecase']).agg(customers = ('customer_id', 'nunique')).reset_index()
df_customer_use_case

,usecase,customers
0,office,21622601


### Academic Urban Commuters

In [104]:
def generate_monthly_queries(start_date, end_date):
    queries = []
    current_date = datetime.strptime(start_date, '%Y%m%d')
    end_date = datetime.strptime(end_date, '%Y%m%d')
    
    while current_date <= end_date:
        month_start = current_date.replace(day=1)
        next_month_start = (month_start + timedelta(days=32)).replace(day=1)
        month_end = next_month_start - timedelta(days=1)
        
        month_start_str = month_start.strftime('%Y%m%d')
        month_end_str = month_end.strftime('%Y%m%d')
        
        query = f"""
            SELECT
                customer_id,
                usecase
            FROM 
                (    
                    (
                    SELECT  
                        customer_id,
                        pickup_usecase.usecase
                    FROM
                        orders.order_logs_snapshot ols 
                    JOIN 
                        experiments_internal.combined_usecase_hex_10_level pickup_usecase
                        ON pickup_location_hex_10 = pickup_usecase.hex_10
                        AND usecase IN ('educational')
                        
                    WHERE 
                        yyyymmdd >= '{month_start_str}'
                        AND yyyymmdd <= '{month_end_str}'
                    )
                UNION
                    (
                    SELECT  
                        customer_id,
                        drop_usecase.usecase 
                    FROM
                        orders.order_logs_snapshot ols 
                    JOIN 
                        experiments_internal.combined_usecase_hex_10_level drop_usecase
                        ON drop_location_hex_10 = drop_usecase.hex_10
                        AND usecase IN ('educational')
                        
                    WHERE 
                        yyyymmdd >= '{month_start_str}'
                        AND yyyymmdd <= '{month_end_str}'
                    )
                UNION 
                    (
                    SELECT 
                        identity customer_id,
                        'educational' usecase
                    FROM 
                        canonical.clevertap_customer_fare_estimate
                    WHERE 
                        yyyymmdd >= '{month_start_str}'
                        AND yyyymmdd <= '{month_end_str}'
                        AND (pickup_address IS NOT NULL OR drop_address IS NOT NULL)
                        AND 
                            (
                            regexp_like(lower(pickup_address), 'school|university|college|institute|academy|elementary|primary|secondary|kindergarten') --campus
                            OR
                            regexp_like(lower(drop_address), 'school|university|college|institute|academy|elementary|primary|secondary|kindergarten') --campus
                            )
                    GROUP BY 1
                    )
                UNION 
                    (
                    SELECT 
                        id customer_id,
                        'educational' usecase
                    FROM 
                        raw_internal.kafka_info_customer_installed_apps_v1
                    WHERE 
                        yyyymmdd >= '{month_start_str}'
                        AND yyyymmdd <= '{month_end_str}'
                        AND regexp_like(lower(data), 'aakash|english|classes|brainly|byju|byte|caclubindia|cbse|chegg|study|colegeduniya|collegedekho|course|coursera|diksha|doubtnut|duolingo|e-pg|pathshala|edx|embibe|fiitjee|geeks|goodreads|classroom|happify|icai|bos|ignou|e-content|khan|academy|meetup|meritnation|math|moodle|mycbseguide|ncert|physics|quizlet|shiksha|swayam|toppr|udemy|unacademy|vedantu|vidyakul|whitehat')
                    GROUP BY 1
                    )
                )
            GROUP BY 1,2
        """
        queries.append(query)
        current_date = next_month_start
    
    return queries

In [107]:
monthly_queries = generate_monthly_queries(start_date, end_date)

# Execute each query and collect the results
df_list = [pd.read_sql(query, connection) for query in monthly_queries]

# Concatenate all dataframes and drop duplicates
df_customer_usecase = pd.concat(df_list).drop_duplicates()

# Group by usecase and count unique customer_ids
df_customer_use_case = df_customer_usecase.groupby(['usecase']).agg(customers = ('customer_id', 'nunique')).reset_index()
df_customer_use_case

### Users spending power

In [30]:
## base 

customer_income_query = f"""

    WITH ao_base AS (
        
        SELECT
            user_id,
            MAX(platform) platform
        FROM 
            canonical.clevertap_customer_order_activity 
        WHERE 
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
        GROUP BY 1
        ),
        
        iallocator_customer_segments AS (
        
        SELECT
            customer_id,
            taxi_income_segment
        FROM 
            datasets.iallocator_customer_segments
        WHERE 
            run_date = DATE_FORMAT(CURRENT_DATE - INTERVAL '5' DAY, '%Y-%m-%d')
        )
        
        SELECT
            '{start_date}' || ' - ' || '{end_date}' AS date_range,
            SUM(customers) OVER(PARTITION BY col)active_customers,
            income_segment,
            customers,
            TRY(customers*100.00/SUM(customers) OVER(PARTITION BY col)) "% distribution"
        FROM
            (
            SELECT
                CASE 
                WHEN taxi_income_segment IS NOT NULL THEN taxi_income_segment
                WHEN platform = 'iOS' THEN 'HIGH_INCOME'
                ELSE 'UNKNOWN' END AS income_segment,
                'dummy' col,
                COUNT(DISTINCT ao_base.user_id) customers
            FROM 
                ao_base
            LEFT JOIN 
                iallocator_customer_segments
                ON user_id = customer_id
            GROUP BY 1
            )
"""

df_customer_income = pd.read_sql(customer_income_query, connection)
df_customer_income

,date_range,active_customers,income_segment,customers,% distribution
0,20240401 - 20240630,36599809,HIGH_INCOME,13356654,36.49
1,20240401 - 20240630,36599809,UNKNOWN,10750773,29.37
2,20240401 - 20240630,36599809,MEDIUM_INCOME,10573301,28.89
3,20240401 - 20240630,36599809,LOW_INCOME,1919081,5.24


### Users exposed to e-commerce

In [44]:
## base 

e_commerce_query = f"""

    SELECT 
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        'ECOMMERCE' usecase,
        COUNT(DISTINCT id) customers
    FROM 
        raw_internal.kafka_info_customer_installed_apps_v1
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND regexp_like(lower(data), 'ajio|aliexpress|amazon|factory|dmart|ebay|firstcry|flipkart|h&m|instashop|jiomart|koovs|lenskart|lifestyle|limeroad|medlife|meesho|myntra|netmeds|nykaa|mall|pepperfry|pharmeasy|shein|shopsy|cliq|souled|westside|zara|zivame')
    GROUP BY 1,2
            
"""

df_ecommerce = pd.read_sql(e_commerce_query, connection)
df_ecommerce

,date_range,usecase,customers
0,20240401 - 20240630,ECOMMERCE,25924340


### On-demand delivery service consumers

In [48]:
## base 

On_Demand_Delivery_query = f"""

    SELECT 
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        'On-Demand Delivery' usecase,
        COUNT(DISTINCT id) customers
    FROM 
        raw_internal.kafka_info_customer_installed_apps_v1
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND regexp_like(lower(data), 'delight|pizza|dunzo|foodpanda|fresh|kfc|licious|mcdonald|starbucks|swiggy|eats|zomato|big|bigbasket|blinkit|grofers|delivery|spencers|zepto|apollo|medibuddy|medplus')
        AND NOT regexp_like(lower(data), 'driver')
    GROUP BY 1,2
            
"""

df_OnDemand_Delivery = pd.read_sql(On_Demand_Delivery_query, connection)
df_OnDemand_Delivery

,date_range,usecase,customers
0,20240401 - 20240630,On-Demand Delivery,17254968


### Travellers

In [13]:
## base 

Travellers_query = f"""
    /*
    SELECT 
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        'Travellers' usecase,
        COUNT(DISTINCT id) customers
    FROM 
        raw_internal.kafka_info_customer_installed_apps_v1
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND regexp_like(lower(data), 'abhibus|air|booking|cleartrip|yatra|easemytrip|expedia|flight|goibibo|indigo|rail|ixigo|kayak|makemytrip|mparivahan|oyo|redbus|skyscanner|spicejet|train|tripadvisor|trivago|vistara|wego|zoomcar')
    GROUP BY 1,2
    */

    WITH multi_cities AS (

    SELECT 
        user_id,
        COUNT(DISTINCT city) cities
    FROM 
        pricing.fare_estimates_enriched
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND city IS NOT NULL 
        AND city != ''
    GROUP BY 1
    HAVING COUNT(DISTINCT city) > 1
    )


    SELECT
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        'Travellers' usecase,
        COUNT(DISTINCT user_id) customers
    FROM 
        multi_cities
    
"""

df_Travellers = pd.read_sql(Travellers_query, connection)
df_Travellers

,date_range,usecase,customers
0,20240401 - 20240630,Travellers,4283473


### Online Gaming and Sports Betting Customers

In [83]:
## base 

gamers_query = f"""

    SELECT 
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        'Online Gaming and Sports Betting' usecase,
        COUNT(DISTINCT id) customers
    FROM 
        raw_internal.kafka_info_customer_installed_apps_v1
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND regexp_like(lower(data), 'rummy|cricbuzz|cricplay|dream|espn|fancode|fanfight|fantasy|mlb|my11circle|myfab11|myteam11|nba|nfl|nhl|games|league|vision11|sports|ball|pool|afterlight|among|asphalt|duty|candy|chess|clash|football|fire|max|racing|ludo|militia|pubg|surfers|run')
    GROUP BY 1,2
            
"""

df_gamers_query = pd.read_sql(gamers_query, connection)
df_gamers_query

,date_range,usecase,customers
0,20240401 - 20240630,Online Gaming and Sports Betting,12368065


## Fintech Customers

In [85]:
## base 

fintech_query = f"""

    SELECT 
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        'Fintech Customers' usecase,
        COUNT(DISTINCT id) customers
    FROM 
        raw_internal.kafka_info_customer_installed_apps_v1
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND regexp_like(lower(data), 'angel|money|groww|hdfc|icici|zerodha|kotak|partner|nps|sbi|yono|sharekhan|upstox|axis|bhim|cred|freecharge|pay|mobikwik|paytm|phonepe|finance')
    GROUP BY 1,2
            
"""

df_fintech = pd.read_sql(fintech_query, connection)
df_fintech

,date_range,usecase,customers
0,20240401 - 20240630,Fintech Customers,22775276


### Job seekers and career professionals

In [100]:
## base 

job_seekers_query = f"""

    SELECT 
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        'Job seekers' usecase,
        COUNT(DISTINCT id) customers
    FROM 
        raw_internal.kafka_info_customer_installed_apps_v1
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '20240727'  --'{end_date}'
        AND regexp_like(lower(data), 'apna|foundit|freelancer|glassdoor|indeed|naukri|naukrigulf|shine')
    GROUP BY 1,2
            
"""

df_job_seekers = pd.read_sql(job_seekers_query, connection)
df_job_seekers

,date_range,usecase,customers


### Online Daters

In [97]:
## base 

OnlineDaters_query = f"""

    SELECT 
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        'Online Daters' usecase,
        COUNT(DISTINCT id) customers
    FROM 
        raw_internal.kafka_info_customer_installed_apps_v1
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND regexp_like(lower(data), 'aisle|bumble|bagel|happn|hinge|okcupid|quackquack|tinder|trulymadly|woo')
    GROUP BY 1,2
            
"""

df_OnlineDaters = pd.read_sql(OnlineDaters_query, connection)
df_OnlineDaters

,date_range,usecase,customers
0,20240401 - 20240630,Online Daters,1188193
